# FloodCastBench - the coarse-resolution regions (Colab)

The main notebook trains on Australia and tests on the UK at **30 m**. That uses about half of FloodCastBench. This notebook
covers the rest, following the paper's own two transfer protocols:

| Experiment | Train | Test | Resolution |
|---|---|---|---|
| `pakistan_mozambique` | Pakistan (2022) | Mozambique (2019) | 480 m (low fidelity) |
| `australia60_uk60` | Australia (2022) | UK (2015) | 60 m (high fidelity) |

**Why these regions were missing before, and what changed.** Their depth rasters are the 30 m simulation resampled to 60 m / 480 m
and cropped, but the terrain, rainfall and land-use files stay on the 30 m grid. The first attempt to align them failed because it
correlated depth with terrain without removing the large-scale trend; a probe that does remove it was validated on the known cases
(Australia 30 m -> offset (0,0), UK 30 m -> (5,5)) and then recovered each coarse region's geometry, cross-checked by independent
signals (see `LAYER_GEOMETRY` in `real_data_loader.py` and `verification_outputs/audit/`).

**Depth only.** At these scales a 128-px patch spans 7.7 km (60 m) or 61 km (480 m), so nearly every patch is "severe" and the
severity label carries no information; severity and the symbolic rules are not evaluated here.

**Not comparable to the paper's tables.** The paper forecasts 19 steps from the initial state over the whole domain; this
notebook forecasts 5 steps (30-minute spacing, 2.5 h) from 5 observed frames on 128-px patches. Everything is judged against
persistence ("copy the last frame"): skill = 1 - MSE_model / MSE_persistence.

The last cells report, for each trained model, skill by lead time and what the model uses (test-time removal of rainfall, terrain
and depth history), because the 30 m audit found the depth skill was driven by rainfall alone.

## Setup

In [ ]:
!pip install -q huggingface_hub rasterio scipy
import warnings
import tensorflow as tf
import rasterio
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)
print("GPU available:", tf.config.list_physical_devices('GPU'))
print("TensorFlow version:", tf.__version__)

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow version: 2.20.0


## Configuration

`EXPERIMENTS` picks what to run. Defaults reproduce the local runs (seed 42, 400 training windows, 40 epochs).

In [ ]:
# CONFIG (overridable)
EPOCHS = 40
N_TRAIN = 400
N_VAL = 150
N_STARTS = 80
SEED = 42
EXPERIMENTS = ["pakistan_mozambique", "australia60_uk60"]

## Pull the data from Hugging Face

Only what the chosen experiments need. **Large:** Pakistan alone is about 9.5 GB (5.8 GB depth + 3.6 GB rainfall); Australia 60 m
about 3.3 GB. Colab's disk is ephemeral, so this re-downloads in a fresh session.

In [ ]:
from huggingface_hub import snapshot_download

PATTERNS = ["Relevant data/Georeferenced files/*"]
if "pakistan_mozambique" in EXPERIMENTS:
    PATTERNS += ["Low-fidelity flood forecasting/480m/*",
                 "Relevant data/DEM/Pakistan_DEM.*", "Relevant data/DEM/Mozambique_DEM.*",
                 "Relevant data/Land use and land cover/Pakistan.*", "Relevant data/Land use and land cover/Mozambique.*",
                 "Relevant data/Rainfall/Pakistan flood/*", "Relevant data/Rainfall/Mozambique flood/*"]
if "australia60_uk60" in EXPERIMENTS:
    PATTERNS += ["High-fidelity flood forecasting/60m/*",
                 "Relevant data/DEM/Australia_DEM.*", "Relevant data/DEM/UK_DEM.*",
                 "Relevant data/Land use and land cover/Australia.*", "Relevant data/Land use and land cover/UK.*",
                 "Relevant data/Rainfall/Australia flood/*", "Relevant data/Rainfall/UK flood/*"]
DATA_ROOT = snapshot_download(repo_id="mahajananhad/floodcastbench", repo_type="dataset", allow_patterns=PATTERNS, max_workers=16)
print("Downloaded to:", DATA_ROOT)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11116 files:   0%|          | 0/11116 [00:00<?, ?it/s]

HTTP Error 429 thrown while requesting GET https://huggingface.co/api/resolve-cache/datasets/mahajananhad/floodcastbench/15a894f18e55580e7de84796d01e5eeec8b95395/Low-fidelity%20flood%20forecasting%2F480m%2FMozambique%2F136200.tif
HTTP Error 429 thrown while requesting GET https://huggingface.co/api/resolve-cache/datasets/mahajananhad/floodcastbench/15a894f18e55580e7de84796d01e5eeec8b95395/Low-fidelity%20flood%20forecasting%2F480m%2FMozambique%2F136500.tif
Rate limited. Waiting 192.0s before retry [Retry 1/5].
Rate limited. Waiting 192.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting GET https://huggingface.co/api/resolve-cache/datasets/mahajananhad/floodcastbench/15a894f18e55580e7de84796d01e5eeec8b95395/Low-fidelity%20flood%20forecasting%2F480m%2FMozambique%2F135000.tif
Rate limited. Waiting 192.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting GET https://huggingface.co/api/resolve-cache/datasets/mahajananhad/floodcastbench/15a894f18e55580e7de84796

Downloaded to: /root/.cache/huggingface/hub/datasets--mahajananhad--floodcastbench/snapshots/15a894f18e55580e7de84796d01e5eeec8b95395


## Core modules (verified locally)

In [ ]:
%%writefile real_data_loader.py
"""
Real FloodCastBench data loader for one region (Stage 1 of the build plan).

Loads a spatial patch + temporal window of depth, DEM, and rainfall for a
single region, aligns rainfall's separate 30-minute calendar-date series to
depth's 5-minute simulation-second series, and derives a severity label from
future depth using the fixed thresholds decided in the project plan.

Verified against the real downloaded files:
  - depth: <region>/<t>.tif, t = unpadded seconds since sim start, step 300s
  - rainfall: Relevant data/Rainfall/<region> flood/<YYYYMMDD>-S<HHMMSS>.tif, step 1800s
  - Australia: depth, rainfall, DEM and land use all share the same 1073x1073 grid.
  - UK: depth is 170x275 but the other layers are 180x285 - a measured (5, 5) pixel
    offset is applied (see LAYER_GEOMETRY).
  - 60 m and 480 m depth rasters (Australia/UK at 60 m; Pakistan/Mozambique at 480 m) are the 30 m
    simulation resampled to 2x2 / 16x16 base-pixel cells and cropped, while DEM / land use / rainfall
    stay on the 30 m base grid. LAYER_GEOMETRY records each combination's scale and crop offset
    (recovered from the data and cross-checked, see its comment); DEM / rainfall are block-averaged
    to the depth cells. A combination with no registered geometry is refused, never guessed.
    (An earlier version of this loader claimed all layers matched - checked only on Australia - and
    later refused Pakistan/Mozambique as "not alignable" - a failure of the first search, not a
    property of the data.)

ASSUMPTION (not confirmed by any embedded metadata - no TIFF tag or readme ties
depth's t=0 to a real calendar date): rainfall's first file corresponds to
depth's t=0. Evidence for this: for all four events the rainfall series starts on the
paper's stated event start date, and its duration equals the depth duration to the second
(e.g. Australia: 481 files at 1800 s = 10 days = 864000 s of depth at 300 s). Date-level
agreement is confirmed; the hour-level start is inferred, not verified.
"""
import os
import re
import glob
from datetime import datetime

import numpy as np
import rasterio

DEPTH_STEP_SECONDS = 300
RAINFALL_STEP_SECONDS = 1800

# Pixel offset (rows, cols) of the depth grid inside the DEM / land-use / rainfall grid,
# for regions where those layers are larger than the depth raster. Measured, not assumed:
# UK depth is 170x275 but its DEM/land-use/rainfall are 180x285. Both an independent
# depth-vs-DEM correlation search (best at dy=5,dx=5, r=-0.27 vs a max of +0.03 over all
# 121 candidate offsets) and a wet-pixels-on-baseline-water enrichment search (~5x at
# dy=5..6,dx=5..6) peak there - consistent with a centered 5px border crop on each side.
# Australia's layers all match the depth grid exactly (verified) so it needs no entry.
#
# Geometry of EVERY (region, resolution) relative to the 30 m base grid that the DEM / land use / rainfall live on:
#   (scale, (row_offset, col_offset))  = base pixels per depth cell, and the base-pixel position of depth cell (0, 0).
# The 60 m and 480 m depth rasters are the 30 m results resampled to 2x2 and 16x16 base-pixel cells and CROPPED (the paper:
# "resampled ... using bilinear interpolation"). Each geometry below was recovered from the data with a probe that was first
# validated on the known cases (Australia 30 m -> (0,0), UK 30 m -> (5,5)), and cross-checked by independent signals:
#   Mozambique 480 m  (16, (150, 50))  terrain-structure r=-0.27 (5.2 robust-sd), DEM NoData all outside the crop
#   Pakistan   480 m  (16, (42, 178))  terrain-structure r=-0.09 (5.5 robust-sd) AND baseline-water r=+0.10 agree; NoData outside
#   UK          60 m  (2,  (5, 5))     terrain-structure r=-0.44 at (5..6, 5..6)
#   Australia   60 m  (2,  (0, 0))     terrain-structure r=-0.26 at (0..1, 0..1)
# Precision is about +-2 base pixels (60 m), i.e. < 6% of a 480 m cell. Scripts and logs: verification_outputs/audit/.
# (An earlier version of this loader refused Pakistan and Mozambique as "not alignable"; that was a failure of the first
# search - it did not remove the large-scale terrain trend - not a property of the data.)
LAYER_GEOMETRY = {
    ("Australia", 30): (1, (0, 0)), ("UK", 30): (1, (5, 5)),
    ("Australia", 60): (2, (0, 0)), ("UK", 60): (2, (5, 5)),
    ("Pakistan", 480): (16, (42, 178)), ("Mozambique", 480): (16, (150, 50)),
}
LAYER_OFFSETS = {"UK": (5, 5)}  # kept for older callers: 30 m offsets only

# The DEM / rainfall rasters carry the float32 NoData sentinel (-3.4028235e38): the UK layers have a padding ring of it
# outside the depth-aligned crop. A window that touched it would feed ~1e38 into the network (and the flat-DEM filter
# in windows.py would count such a patch as "valid", its DEM std being enormous). None of the windows this project
# uses touch it (verification_outputs/audit/investigate_uk_nodata.py), but the loader now refuses instead of trusting that.
NODATA_BELOW = -1e30


def has_nodata(arr: np.ndarray) -> bool:
    """True if `arr` holds the NoData sentinel or any non-finite value."""
    return bool((arr < NODATA_BELOW).any() or not np.isfinite(arr).all())

# Fixed absolute depth thresholds (meters), decided in the project plan.
SEVERITY_THRESHOLDS = [0.3, 1.0, 2.0]  # -> none / minor / moderate / severe
SEVERITY_CLASSES = ["none", "minor", "moderate", "severe"]


def depth_to_severity(max_depth_m: float) -> int:
    """Bin a max-depth value (meters) into one of the 4 fixed severity classes."""
    idx = 0
    for t in SEVERITY_THRESHOLDS:
        if max_depth_m >= t:
            idx += 1
    return idx  # 0=none, 1=minor, 2=moderate, 3=severe


class RegionData:
    """Indexes one region's depth/rainfall/DEM files without loading pixel data."""

    def __init__(self, root_dir: str, region: str, fidelity_dir: str):
        """
        root_dir: path to the extracted FloodCastBench folder
        region: e.g. "Australia"
        fidelity_dir: e.g. "High-fidelity flood forecasting/30m"
        """
        self.root_dir = root_dir
        self.region = region

        depth_dir = os.path.join(root_dir, fidelity_dir, region)
        depth_files = glob.glob(os.path.join(depth_dir, "*.tif"))
        if not depth_files:
            raise FileNotFoundError(f"No depth TIFFs found in {depth_dir}")

        def depth_seconds(path):
            name = os.path.splitext(os.path.basename(path))[0]
            m = re.match(r"^(\d+)$", name)
            if not m:
                raise ValueError(f"Expected purely numeric depth filename, got: {path}")
            return int(m.group(1))

        self.depth_files = sorted(depth_files, key=depth_seconds)
        self.depth_seconds = [depth_seconds(p) for p in self.depth_files]

        step = self.depth_seconds[1] - self.depth_seconds[0]
        for i in range(1, len(self.depth_seconds)):
            gap = self.depth_seconds[i] - self.depth_seconds[i - 1]
            if gap != step:
                raise ValueError(
                    f"Depth sequence is not evenly spaced at index {i}: "
                    f"gap={gap}s, expected {step}s"
                )
        self.depth_step_seconds = step

        rain_dir = os.path.join(root_dir, "Relevant data", "Rainfall", f"{region} flood")
        rain_files = glob.glob(os.path.join(rain_dir, "*.tif"))
        if not rain_files:
            raise FileNotFoundError(f"No rainfall TIFFs found in {rain_dir}")

        def rain_datetime(path):
            name = os.path.splitext(os.path.basename(path))[0]
            m = re.match(r"^(\d{8})-S(\d{6})$", name)
            if not m:
                raise ValueError(f"Unexpected rainfall filename format: {path}")
            return datetime.strptime(m.group(1) + m.group(2), "%Y%m%d%H%M%S")

        rain_with_dt = sorted(((rain_datetime(p), p) for p in rain_files), key=lambda x: x[0])
        self.rain_files = [p for _, p in rain_with_dt]
        self.rain_start = rain_with_dt[0][0]
        rain_step = (rain_with_dt[1][0] - rain_with_dt[0][0]).total_seconds()
        for i in range(1, len(rain_with_dt)):
            gap = (rain_with_dt[i][0] - rain_with_dt[i - 1][0]).total_seconds()
            if gap != rain_step:
                raise ValueError(f"Rainfall sequence is not evenly spaced at index {i}")
        self.rain_step_seconds = rain_step

        self.dem_path = os.path.join(root_dir, "Relevant data", "DEM", f"{region}_DEM.tif")
        if not os.path.isfile(self.dem_path):
            raise FileNotFoundError(self.dem_path)

        self.landuse_path = os.path.join(root_dir, "Relevant data", "Land use and land cover", f"{region}.tif")
        if not os.path.isfile(self.landuse_path):
            raise FileNotFoundError(self.landuse_path)

        with rasterio.open(self.depth_files[0]) as src:
            self.grid_height, self.grid_width = src.height, src.width

        m = re.search(r"(\d+)m$", fidelity_dir.rstrip("/"))
        if not m:
            raise ValueError(
                f"Cannot parse pixel size from fidelity_dir={fidelity_dir!r} "
                f"(expected it to end in e.g. '30m')"
            )
        self.pixel_size_m = float(m.group(1))

        key = (region, int(round(self.pixel_size_m)))
        if key not in LAYER_GEOMETRY:
            raise ValueError(
                f"{region} at {self.pixel_size_m:.0f} m has no registered layer geometry in LAYER_GEOMETRY - windows read "
                f"at the same (row, col) would cover different geography. Refusing to load rather than return misaligned data."
            )
        self.layer_scale, self.layer_offset = LAYER_GEOMETRY[key]
        dy, dx = self.layer_offset
        s = self.layer_scale
        for label, path in (("DEM", self.dem_path), ("land use", self.landuse_path),
                            ("rainfall", self.rain_files[0])):
            with rasterio.open(path) as src:
                layer_h, layer_w = src.height, src.width
            if dy + s * self.grid_height > layer_h or dx + s * self.grid_width > layer_w:
                raise ValueError(
                    f"{region}: {label} grid is {layer_h}x{layer_w}, too small for the registered geometry "
                    f"(scale {s}, offset {self.layer_offset}) of a {self.grid_height}x{self.grid_width} depth grid."
                )

    def _layer_window(self, row: int, col: int, patch_size: int):
        """Window (in 30 m base pixels) into the DEM / land-use / rainfall grids for a depth-grid (row, col) patch."""
        dy, dx = self.layer_offset
        s = self.layer_scale
        return ((dy + s * row, dy + s * (row + patch_size)), (dx + s * col, dx + s * (col + patch_size)))

    def _read_layer(self, path: str, row: int, col: int, patch_size: int, nearest: bool = False) -> np.ndarray:
        """Read a base-grid layer for a depth-grid patch, reduced to (patch, patch): block-mean over the cell's
        scale x scale base pixels (nearest = the cell-centre pixel, for categorical land use)."""
        s = self.layer_scale
        with rasterio.open(path) as src:
            a = src.read(1, window=self._layer_window(row, col, patch_size), boundless=True, fill_value=0)
        if s == 1:
            return a
        a = a.reshape(patch_size, s, patch_size, s)
        if nearest:
            return a[:, s // 2, :, s // 2]
        return a.astype(np.float64).mean(axis=(1, 3)).astype(np.float32)

    def load_dem_patch(self, row: int, col: int, patch_size: int) -> np.ndarray:
        return self._read_layer(self.dem_path, row, col, patch_size).astype(np.float32)

    def rainfall_index_for_depth_second(self, depth_second: int) -> int:
        """Map a depth timestep (seconds since sim start) to its rainfall file index,
        holding each 30-min rainfall value constant across the depth steps inside it."""
        idx = int(depth_second // self.rain_step_seconds)
        return min(idx, len(self.rain_files) - 1)

    def window_frame_indices(self, start_idx: int, t_in: int, horizon: int, stride: int = 1):
        """Depth-file indices for one window: (history_indices, future_indices).

        History is start_idx, start_idx+stride, ...; the forecast continues at the same
        stride. stride=1 is the native 5-minute step; stride=6 is 30 minutes, which is
        rainfall's native resolution (at stride 1, five history frames span only 25
        minutes, so the rainfall channel is essentially constant across the sequence).
        The severity scan in windows.py uses this same method so scan and loader can
        never disagree about which frames a window covers."""
        history = [start_idx + stride * k for k in range(t_in)]
        future = [start_idx + stride * (t_in + k) for k in range(horizon)]
        if future[-1] >= len(self.depth_files):
            raise ValueError(
                f"start_idx={start_idx} t_in={t_in} horizon={horizon} stride={stride} runs past "
                f"the end of the depth sequence ({len(self.depth_files)} files)"
            )
        return history, future

    def max_start_index(self, t_in: int, horizon: int, stride: int = 1) -> int:
        """Largest start_idx for which a whole window still fits."""
        return len(self.depth_files) - 1 - stride * (t_in + horizon - 1)

    def load_patch_sequence(self, row: int, col: int, patch_size: int,
                             start_idx: int, t_in: int, horizon: int, stride: int = 1):
        """Read one (row, col) patch across the window described by window_frame_indices():
        the history frames are the model input, the future frames are both the regression
        targets and the source of the severity label.

        Returns: input_sequence (t_in, H, W, 3) [depth, DEM, rainfall],
                 depth_targets (horizon, H, W),
                 severity_label (int, 0-3), max_future_depth (float)
        """
        window = ((row, row + patch_size), (col, col + patch_size))
        history_idx, future_idx = self.window_frame_indices(start_idx, t_in, horizon, stride)

        dem_patch = self.load_dem_patch(row, col, patch_size)

        depth_frames = []
        rainfall_frames = []
        for i in history_idx:
            with rasterio.open(self.depth_files[i]) as src:
                depth_frames.append(src.read(1, window=window, boundless=True, fill_value=0).astype(np.float32))
            rain_idx = self.rainfall_index_for_depth_second(self.depth_seconds[i])
            rainfall_frames.append(self._read_layer(self.rain_files[rain_idx], row, col, patch_size).astype(np.float32))

        depth_hist = np.stack(depth_frames)          # (t_in, H, W)
        rainfall_hist = np.stack(rainfall_frames)     # (t_in, H, W)
        dem_seq = np.broadcast_to(dem_patch, depth_hist.shape)  # (t_in, H, W)
        input_sequence = np.stack([depth_hist, dem_seq, rainfall_hist], axis=-1)  # (t_in, H, W, 3)

        depth_targets = []
        for i in future_idx:
            with rasterio.open(self.depth_files[i]) as src:
                depth_targets.append(src.read(1, window=window, boundless=True, fill_value=0).astype(np.float32))
        depth_targets = np.stack(depth_targets)  # (horizon, H, W)

        for what, arr in (("depth history", depth_hist), ("rainfall", rainfall_hist),
                          ("DEM", dem_patch), ("depth targets", depth_targets)):
            if has_nodata(arr):
                raise ValueError(
                    f"{self.region}: {what} window at depth-grid (row={row}, col={col}) contains NoData/NaN pixels "
                    f"(the layers have a NoData padding ring outside the aligned crop). Refusing to return it."
                )

        max_future_depth = float(depth_targets.max())
        severity_label = depth_to_severity(max_future_depth)

        return input_sequence, depth_targets, severity_label, max_future_depth

    def load_landuse_patch(self, row: int, col: int, patch_size: int) -> np.ndarray:
        """Real Esri/Impact Observatory Sentinel-2 land-cover codes (verified against
        this exact product's published legend): 1=Water, 2=Trees, 4=Flooded Vegetation,
        5=Crops, 7=Built Area, 8=Bare Ground, 9=Snow/Ice, 10=Clouds, 11=Rangeland.
        At 60 m / 480 m the class of each cell's centre pixel is returned (nearest neighbour)."""
        return self._read_layer(self.landuse_path, row, col, patch_size, nearest=True).astype(np.int32)


Writing real_data_loader.py


In [ ]:
%%writefile capsule_layers.py
"""
Capsule Network layers (Sabour, Frosst & Hinton 2017, arXiv:1710.09829) for the
severity-classification branch: PrimaryCaps -> routing-by-agreement -> per-class
output capsules, whose vector lengths give the 4-way severity probability.

Kept as plain tf.keras.layers.Layer subclasses (not a separate framework) so the
whole model - depth-forecasting branch and severity branch alike - trains as one
TensorFlow/Keras graph: gradients from the severity loss reach back through
routing, through PrimaryCaps, into the SAME shared CNN encoder the LSTM branch
uses. This is the piece that was previously impossible when the capsule layer
lived in a separate PyTorch model (see project plan, Stage 3).
"""
import tensorflow as tf
from tensorflow.keras import layers


def squash(vectors: tf.Tensor, axis: int = -1, epsilon: float = 1e-7) -> tf.Tensor:
    """Capsule nonlinearity: shrinks short vectors to (near) zero and long vectors
    to just under length 1, so a capsule's output length can be read as a
    probability. squash(s) = (|s|^2 / (1 + |s|^2)) * (s / |s|)."""
    squared_norm = tf.reduce_sum(tf.square(vectors), axis=axis, keepdims=True)
    scale = squared_norm / (1.0 + squared_norm) / tf.sqrt(squared_norm + epsilon)
    return scale * vectors


class PrimaryCaps(layers.Layer):
    """First capsule layer: a Conv2D projects a spatial feature map into
    (num_capsules * capsule_dim) channels, then reshapes each spatial location's
    channel group into one capsule vector, and squashes it."""

    def __init__(self, num_capsules: int, capsule_dim: int, kernel_size: int = 3,
                 strides: int = 2, **kwargs):
        super().__init__(**kwargs)
        self.num_capsules = num_capsules
        self.capsule_dim = capsule_dim
        self.conv = layers.Conv2D(
            num_capsules * capsule_dim, kernel_size, strides=strides,
            padding="same", activation=None, name="primary_caps_conv",
        )

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """inputs: (batch, H, W, C) spatial feature map.
        Returns: (batch, n_primary_caps, capsule_dim) - flattened over space and
        the num_capsules groups, each a squashed vector."""
        x = self.conv(inputs)  # (batch, h, w, num_capsules * capsule_dim)
        batch = tf.shape(x)[0]
        h, w = x.shape[1], x.shape[2]
        n_primary_caps = h * w * self.num_capsules
        x = tf.reshape(x, (batch, n_primary_caps, self.capsule_dim))
        return squash(x, axis=-1)


class RoutingCaps(layers.Layer):
    """Routing-by-agreement between a PrimaryCaps layer and this layer's output
    capsules (one per class). Each input capsule casts a weighted "vote" for each
    output capsule (via a learned transform W); routing coefficients are refined
    over `n_routing` iterations based on how well each vote agrees with the
    current output capsule (dot-product agreement), per Sabour et al.'s
    Algorithm 1 - this is what "routing-by-agreement" in the architecture
    diagram refers to, not a fixed pooling/summation."""

    def __init__(self, num_output_capsules: int, output_capsule_dim: int,
                 n_routing: int = 3, **kwargs):
        super().__init__(**kwargs)
        self.num_output_capsules = num_output_capsules
        self.output_capsule_dim = output_capsule_dim
        self.n_routing = n_routing

    def build(self, input_shape):
        n_input_caps, input_capsule_dim = input_shape[1], input_shape[2]
        # W: one learned transform matrix per (input capsule, output capsule) pair
        self.W = self.add_weight(
            name="routing_weights",
            shape=(n_input_caps, self.num_output_capsules, self.output_capsule_dim, input_capsule_dim),
            initializer="glorot_uniform",
            trainable=True,
        )

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """inputs: (batch, n_input_caps, input_capsule_dim)
        Returns: (batch, num_output_capsules, output_capsule_dim)"""
        batch = tf.shape(inputs)[0]
        n_input_caps = inputs.shape[1]

        # votes[b,i,j,:] = W[i,j] @ inputs[b,i,:]  -> (batch, n_in, n_out, out_dim).
        # einsum broadcasts the batch dim automatically (W has none) - no manual
        # tiling needed, which is exactly where the shape bug above came from.
        votes = tf.einsum("ijod,bid->bijo", self.W, inputs)

        routing_logits = tf.zeros((batch, n_input_caps, self.num_output_capsules))
        votes_stopped = tf.stop_gradient(votes)

        outputs = None
        for i in range(self.n_routing):
            routing_weights = tf.nn.softmax(routing_logits, axis=2)  # agreement coefficients c_ij, sum over j=1
            routing_weights = routing_weights[..., tf.newaxis]        # (b, n_in, n_out, 1)

            use_votes = votes if i == self.n_routing - 1 else votes_stopped
            weighted_sum = tf.reduce_sum(routing_weights * use_votes, axis=1)  # (b, n_out, out_dim)
            outputs = squash(weighted_sum, axis=-1)

            if i < self.n_routing - 1:
                agreement = tf.reduce_sum(votes_stopped * outputs[:, tf.newaxis, :, :], axis=-1)  # (b, n_in, n_out)
                routing_logits = routing_logits + agreement

        return outputs


def margin_loss(labels: tf.Tensor, lengths: tf.Tensor, m_plus: float = 0.9,
                m_minus: float = 0.1, down_weight: float = 0.5) -> tf.Tensor:
    """The margin loss from Sabour et al. 2017, eq. 4 - the loss capsule lengths were designed for.
    labels: (batch,) integer class ids; lengths: (batch, n_classes) in [0, 1).

    Why not softmax cross-entropy on the lengths: lengths are bounded in [0, 1), so a softmax over
    them can never exceed ~e/(e+3) = 0.475 for one of 4 classes - the loss floors near 0.74 and the
    gradient signal is weak (measured locally: predictions collapsed to one class). The margin loss
    instead pushes the true class's length above m_plus and every other class's below m_minus."""
    targets = tf.one_hot(labels, depth=lengths.shape[-1], dtype=lengths.dtype)
    present = targets * tf.square(tf.nn.relu(m_plus - lengths))
    absent = down_weight * (1.0 - targets) * tf.square(tf.nn.relu(lengths - m_minus))
    return tf.reduce_mean(tf.reduce_sum(present + absent, axis=-1))


def capsule_lengths(output_capsules: tf.Tensor, epsilon: float = 1e-7) -> tf.Tensor:
    """(batch, n_classes, dim) -> (batch, n_classes) vector lengths, read as
    per-class confidence. Softmax over these gives a proper severity probability
    distribution for use with standard categorical cross-entropy."""
    return tf.sqrt(tf.reduce_sum(tf.square(output_capsules), axis=-1) + epsilon)


Writing capsule_layers.py


In [ ]:
%%writefile model.py
"""
CNN -> LSTM encoder-decoder for spatiotemporal flood depth forecasting.

Input: (batch, T_in, H, W, C=3) - depth[t], static DEM, rainfall[t] per historical timestep.
Output: (batch, T_out, H, W) depth regression only.

This is the depth-forecasting branch only, per the current (Capsule Network + Symbolic
Rule Layer) architecture - the per-pixel dry/flooded classifier this module used to carry
predates that pivot and has been removed. Severity classification now lives entirely in
the separate Capsule Network branch (build_capsule_head, added in Stage 3), which reads
the same shared CNN-encoder features but is trained/evaluated independently of this module.

Architecture follows the dossier's Section 4.5 illustrative design (time-distributed CNN
encoder -> LSTM temporal encoder-decoder -> CNN decoder), itself following the
time-distributed-CNN-then-LSTM pattern from Malik et al. 2024 and the CNN/LSTM-fused-at-
bottleneck pattern from Wang et al. 2026 (see research dossier, Section 2.B).

CO mapping:
  CO1 - dense_bottleneck: Dense layer projecting CNN features to the LSTM's input vector
  CO2 - severity classification head: now the Capsule Network (see Stage 3), not this module
  CO3 - cnn_encoder_*/cnn_decoder_*: convolutional spatial feature extraction/reconstruction
  CO4 - dropout/optimizer/learning_rate exposed as build_models() parameters
  CO5 - encoder_lstm/decoder_lstm: the temporal encoder-decoder generating the forecast sequence
"""

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from capsule_layers import PrimaryCaps, RoutingCaps, capsule_lengths

PATCH_SIZE = 128  # H = W = 128, per dossier Section 4.5
N_CHANNELS = 3    # depth, DEM, rainfall
NUM_SEVERITY_CLASSES = 4  # none / minor / moderate / severe, see real_data_loader.py

# Fixed (non-learned) scales so raw physical inputs feed the CNN sanely. Without these,
# DEM values (tens to hundreds of metres) swamp rainfall (0-9 mm) and depth (0-18 m).
DEPTH_SCALE_M = 5.0     # input depth is divided by this; the decoder multiplies back to metres
DEM_SCALE_M = 50.0      # DEM is made patch-relative (minus its own mean) then divided by this
RAIN_SCALE_MM = 5.0
DELTA_SCALE_M = 0.05    # the decoder's unit of "change from the last observed frame" is 5 cm


class InputNormalizer(layers.Layer):
    """Fixed per-channel scaling applied INSIDE the model, so every caller (training,
    predict_severity, Grad-CAM) keeps passing raw physical units and getting metres back.

    The DEM channel is patch-relative (mean removed) rather than absolute elevation: absolute
    elevation is a region-specific fingerprint that can't transfer to a new region, while
    relative relief (slopes, valleys) can - which matters for the cross-regional test."""

    def call(self, x):  # x: (batch, T, H, W, 3) raw units
        depth = x[..., 0:1] / DEPTH_SCALE_M
        dem = x[..., 1:2]
        dem_rel = (dem - tf.reduce_mean(dem, axis=(2, 3), keepdims=True)) / DEM_SCALE_M
        rain = x[..., 2:3] / RAIN_SCALE_MM
        return tf.concat([depth, dem_rel, rain], axis=-1)

OPTIMIZERS = {
    "adam": keras.optimizers.Adam,
    "rmsprop": keras.optimizers.RMSprop,
    "sgd": keras.optimizers.SGD,
}


def build_encoder(hidden_dim: int = 128, dropout: float = 0.3) -> keras.Model:
    """Shared CNN encoder + LSTM temporal encoder. Outputs both the LSTM's final
    states (for the depth-forecasting decoder) AND the raw per-timestep CNN
    feature maps, BEFORE they're flattened into the LSTM - this second output is
    what the Capsule Network (Stage 3) consumes, so both branches genuinely share
    one encoder and both receive real gradients through it, rather than the
    Capsule Network operating on a separately re-computed copy."""
    inputs = keras.Input(shape=(None, PATCH_SIZE, PATCH_SIZE, N_CHANNELS), name="input_sequence")

    x = InputNormalizer(name="input_normalizer")(inputs)
    x = layers.TimeDistributed(layers.Conv2D(16, 3, strides=2, padding="same", activation="relu"), name="cnn_encoder_1")(x)
    x = layers.TimeDistributed(layers.Conv2D(32, 3, strides=2, padding="same", activation="relu"), name="cnn_encoder_2")(x)
    cnn_features = layers.TimeDistributed(layers.Conv2D(64, 3, strides=2, padding="same", activation="relu"), name="cnn_encoder_3")(x)
    # 128x128 -> 16x16x64 after three stride-2 blocks; cnn_features: (batch, T, 16, 16, 64)

    x = layers.TimeDistributed(layers.Flatten())(cnn_features)
    x = layers.TimeDistributed(layers.Dropout(dropout))(x)
    x = layers.TimeDistributed(layers.Dense(hidden_dim, activation="relu"), name="dense_bottleneck")(x)  # CO1

    _outputs, state_h, state_c = layers.LSTM(
        hidden_dim, return_sequences=True, return_state=True, dropout=dropout, name="encoder_lstm"
    )(x)

    return keras.Model(inputs, [state_h, state_c, cnn_features], name="encoder")


def build_decoder_step(hidden_dim: int = 128, dropout: float = 0.3) -> keras.Model:
    """One decode step. The LSTM takes a constant learned "step" input and relies entirely
    on its (state_h, state_c) carried from the previous step to produce a different output
    each time - standard for this kind of generative decoder, and much cheaper than
    re-encoding the full previous frame through a dense layer just to feed it back in.

    PERSISTENCE-ANCHORED (residual): the output is the most recent OBSERVED depth frame plus
    a learned change, not a depth field rebuilt from scratch. The first version rebuilt the
    whole 128x128 field from a 128-number LSTM state with no link to the observed frame, so
    it could never approach a near-exact copy - and a copy of the last frame is already
    accurate to ~2 mm at a 5-minute step, ~13 mm at 30 minutes (measured on real data).
    That, not only "too little data", is why the early predictions were blurry. The final
    conv is zero-initialized, so training starts exactly at the persistence forecast and
    can only improve on it."""
    step_input = keras.Input(shape=(1,), name="step_input")  # constant scalar, e.g. tf.ones((1,1))
    state_h_in = keras.Input(shape=(hidden_dim,), name="state_h")
    state_c_in = keras.Input(shape=(hidden_dim,), name="state_c")
    anchor_in = keras.Input(shape=(PATCH_SIZE, PATCH_SIZE, 1), name="anchor_depth")  # last observed depth, metres
    skip_in = keras.Input(shape=(16, 16, 64), name="skip_features")  # encoder's last-step spatial features

    step_emb = layers.Dense(hidden_dim, name="step_embedding")(step_input)
    step_emb_seq = layers.Reshape((1, hidden_dim))(step_emb)
    lstm_out, state_h, state_c = layers.LSTM(hidden_dim, return_state=True, name="decoder_lstm")(
        step_emb_seq, initial_state=[state_h_in, state_c_in]
    )
    lstm_out = layers.Dropout(dropout)(lstm_out)

    # project back to a 16x16x64 feature map, then upsample with transposed convs (mirrors the encoder)
    x = layers.Dense(16 * 16 * 64, activation="relu", name="decoder_bottleneck")(lstm_out)
    x = layers.Reshape((16, 16, 64))(x)
    # Skip connection: the LSTM state alone is a 128-number summary, so the decoder can express a
    # global drift but has no way to say WHERE anything changes. Concatenating the encoder's own
    # last-step spatial features gives every 16x16 cell access to the local observed structure.
    # (Measured: without this, held-out skill plateaued at exactly +2% from epoch 2 onward, and
    # was the same +2% on a region never seen in training - a constant drift, not learning.)
    x = layers.Concatenate(name="context_plus_skip")([x, skip_in])
    x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu", name="cnn_decoder_1")(x)
    x = layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="relu", name="cnn_decoder_2")(x)
    x = layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="relu", name="cnn_decoder_3")(x)
    # 16x16 -> 32x32 -> 64x64 -> 128x128

    delta = layers.Conv2D(1, 1, activation="linear", kernel_initializer="zeros", name="depth_delta")(x)
    delta = layers.Rescaling(DELTA_SCALE_M, name="delta_to_meters")(delta)
    depth_out = layers.Add(name="anchor_plus_delta")([anchor_in, delta])               # (batch,128,128,1) metres

    return keras.Model(
        [step_input, state_h_in, state_c_in, anchor_in, skip_in],
        [depth_out, state_h, state_c],
        name="decoder_step",
    )


def anchor_from_input(x):
    """Most recent observed depth frame from a raw (batch, T, H, W, 3) input -> (batch, H, W, 1)."""
    return x[:, -1, :, :, 0:1]


def decode_horizon(decoder, enc_out, x, horizon: int, training: bool = False):
    """Runs the residual decoder for `horizon` steps. Returns (batch, horizon, H, W) depth in metres.

    enc_out: the encoder's outputs [state_h, state_c, cnn_features]; x: the SAME raw input batch
    the encoder was run on (its last depth frame is the persistence anchor). Every step is
    anchored to that frame and sees the encoder's last-step spatial features; the LSTM state
    evolves the change."""
    state_h, state_c, cnn_features = enc_out
    anchor_depth = anchor_from_input(x)
    skip = cnn_features[:, -1]
    step_input = tf.ones((tf.shape(state_h)[0], 1))
    frames = []
    sh, sc = state_h, state_c
    for _ in range(horizon):
        depth_out, sh, sc = decoder([step_input, sh, sc, anchor_depth, skip], training=training)
        frames.append(depth_out[..., 0])
    return tf.stack(frames, axis=1)


def build_capsule_head(num_primary_capsules: int = 8, primary_capsule_dim: int = 8,
                        severity_capsule_dim: int = 16, n_routing: int = 3,
                        return_lengths: bool = False) -> keras.Model:
    """Severity-classification branch (Stage 3): PrimaryCaps -> routing-by-agreement
    -> one capsule per severity class. Consumes the LAST timestep of the SAME
    cnn_features the encoder outputs - i.e. the shared encoder's own most-recent
    observed spatial representation, not a separately-computed copy - so this
    branch's gradients flow back into the real shared encoder weights.

    return_lengths=False (default): output softmax(lengths), for use with cross-entropy.
    return_lengths=True: output the raw capsule lengths in [0, 1) for use with
    capsule_layers.margin_loss (the loss the paper designed lengths for). argmax is the
    prediction either way."""
    cnn_feat_input = keras.Input(shape=(16, 16, 64), name="cnn_features_last_step")
    primary = PrimaryCaps(num_primary_capsules, primary_capsule_dim, kernel_size=3, strides=2)(cnn_feat_input)
    severity_caps = RoutingCaps(NUM_SEVERITY_CLASSES, severity_capsule_dim, n_routing=n_routing)(primary)
    lengths = layers.Lambda(capsule_lengths, name="capsule_lengths")(severity_caps)
    if return_lengths:
        return keras.Model(cnn_feat_input, lengths, name="capsule_severity_head")
    probs = layers.Softmax(name="severity_probs")(lengths)
    return keras.Model(cnn_feat_input, probs, name="capsule_severity_head")


def build_models(hidden_dim: int = 128, dropout: float = 0.3):
    return build_encoder(hidden_dim, dropout), build_decoder_step(hidden_dim, dropout)


def forecast(encoder, decoder, input_sequence: np.ndarray, horizon: int, hidden_dim: int = 128):
    """Real forward pass: encode T_in history, then decode `horizon` future frames."""
    x = tf.constant(input_sequence[np.newaxis, ...].astype("float32"))  # (1, T_in, H, W, C)
    enc_out = encoder(x, training=False)

    shapes = {"input": tuple(x.shape), "encoder_state_h": tuple(enc_out[0].shape)}
    frames = decode_horizon(decoder, enc_out, x, horizon, training=False)
    shapes["decoded_frames"] = tuple(frames.shape)
    return frames.numpy()[0], shapes


def dummy_training_step(encoder, decoder, input_sequence: np.ndarray, horizon: int,
                         optimizer_name: str = "adam", learning_rate: float = 1e-3, hidden_dim: int = 128):
    """One real gradient step against random fake targets - proves the loss/optimizer/dropout
    wiring (CO4) is mechanically correct. Not real training; the targets are random."""
    optimizer = OPTIMIZERS[optimizer_name](learning_rate=learning_rate)
    x = tf.constant(input_sequence[np.newaxis, ...].astype("float32"))
    rng = np.random.default_rng(0)
    fake_depth_targets = tf.constant(rng.random((horizon, PATCH_SIZE, PATCH_SIZE)).astype("float32"))

    trainable_vars = encoder.trainable_variables + decoder.trainable_variables

    with tf.GradientTape() as tape:
        enc_out = encoder(x, training=True)
        frames = decode_horizon(decoder, enc_out, x, horizon, training=True)
        loss = tf.reduce_mean(tf.square(frames[0] - fake_depth_targets))

    grads = tape.gradient(loss, trainable_vars)
    n_none = sum(1 for g in grads if g is None)
    optimizer.apply_gradients((g, v) for g, v in zip(grads, trainable_vars) if g is not None)

    return float(loss.numpy()), n_none, len(trainable_vars)


Writing model.py


In [ ]:
%%writefile windows.py
"""
Window scanning and sampling for FloodCastBench regions.

A "window" is one (spatial patch, start time). Severity is computed EXACTLY the way
RegionData.load_patch_sequence computes it - max depth in the patch over the future
frames from RegionData.window_frame_indices - and both use that same method, so a scan
can never disagree with what the loader later returns.

Each scanned window is a tuple:
    (row, col, start_idx, severity, baseline_severity)
where baseline_severity is what you'd get by just applying the severity thresholds to the
max depth of the LAST OBSERVED frame (a "persistence" classifier that needs no model).
Measured on real Australia data, that trivial classifier is 99.9% accurate at a 5-minute
step and 96.7% at a 30-minute stride - so any severity model has to be judged against it.
"""
import numpy as np
import rasterio

from real_data_loader import SEVERITY_CLASSES, RegionData, depth_to_severity, has_nodata


def all_positions(rd: RegionData, patch_size: int):
    """Non-overlapping patch origins that fit entirely inside the depth grid."""
    return [(r, c) for r in range(0, rd.grid_height - patch_size + 1, patch_size)
                   for c in range(0, rd.grid_width - patch_size + 1, patch_size)]


def valid_positions(rd: RegionData, patch_size: int, dem_nodata_std: float = 1.0):
    """Drop patches whose DEM is (near-)flat - those lie outside the simulation domain - and patches whose
    DEM window touches the NoData sentinel (its std is enormous, so the flat-DEM test alone would keep it)."""
    keep = []
    for (r, c) in all_positions(rd, patch_size):
        dem = rd.load_dem_patch(r, c, patch_size)
        if not has_nodata(dem) and dem.std() > dem_nodata_std:
            keep.append((r, c))
    return keep


def start_indices_for(rd: RegionData, count: int, t_in: int, horizon: int, stride: int):
    """`count` start indices spread evenly over every start at which a whole window fits."""
    last = rd.max_start_index(t_in, horizon, stride)
    return np.unique(np.linspace(0, last, count).astype(int))


def scan_windows(rd: RegionData, positions, start_indices, patch_size: int,
                 t_in: int, horizon: int, stride: int):
    out = []
    for start_idx in start_indices:
        history_idx, future_idx = rd.window_frame_indices(int(start_idx), t_in, horizon, stride)
        with rasterio.open(rd.depth_files[history_idx[-1]]) as src:
            last_observed = src.read(1)
        future_max = None
        for i in future_idx:
            with rasterio.open(rd.depth_files[i]) as src:
                frame = src.read(1)
            future_max = frame if future_max is None else np.maximum(future_max, frame)
        for (row, col) in positions:
            sl = (slice(row, row + patch_size), slice(col, col + patch_size))
            out.append((row, col, int(start_idx),
                        depth_to_severity(float(future_max[sl].max())),
                        depth_to_severity(float(last_observed[sl].max()))))
    return out


def class_fractions(windows):
    total = len(windows)
    return {c: sum(1 for w in windows if w[3] == i) / total for i, c in enumerate(SEVERITY_CLASSES)}


def balanced_sample(pool, per_class: int, rng):
    """Up to `per_class` randomly-chosen windows of each severity class."""
    chosen = []
    for cls_idx, cls in enumerate(SEVERITY_CLASSES):
        members = [w for w in pool if w[3] == cls_idx]
        take = min(per_class, len(members))
        if take < per_class:
            print(f"  WARNING: only {len(members)} '{cls}' windows available (wanted {per_class})")
        pick = rng.choice(len(members), size=take, replace=False)
        chosen.extend(members[i] for i in pick)
    return chosen


Writing windows.py


In [ ]:
%%writefile depth_experiment.py
"""Depth-forecast experiments on the COARSE parts of FloodCastBench, following the paper's own two transfer protocols:

  480 m  low-fidelity :  train on Pakistan, test on Mozambique      (the paper's low-fidelity transfer test)
   60 m  high-fidelity:  train on Australia, test on the UK          (the paper's high-fidelity transfer test)

Depth only. At 480 m and 60 m the severity label is degenerate (a 128-px patch covers 61 km / 7.7 km, so nearly every patch is
"severe"), so no severity is scored here. Same model as the 30 m experiment (persistence-anchored CNN-LSTM), same loss
(MSE / persistence MSE), same metric (skill vs persistence, = 1 - MSE_model / MSE_persistence), so results are comparable
in kind. NOT comparable to the paper's tables: those forecast 19 steps from the initial state over the whole domain.

Also reports, for each trained model, (a) skill by lead time and (b) what the model uses (test-time ablations), because the
30 m audit found the depth skill was driven by rainfall and not by the depth history.

Run:  python depth_experiment.py pakistan_mozambique        |   python depth_experiment.py australia60_uk60
"""
import json
import os
import sys
import time

import numpy as np
import rasterio
import tensorflow as tf

import model as flood_model
from model import decode_horizon
from real_data_loader import RegionData
from windows import valid_positions, start_indices_for

PATCH, T_IN, HORIZON, STRIDE = 128, 5, 5, 6          # 6 x 5 min = 30 min: rainfall's native step, same as the 30 m experiment
ROOT = "data/extracted/FloodCastBench"
FID = {30: "High-fidelity flood forecasting/30m", 60: "High-fidelity flood forecasting/60m", 480: "Low-fidelity flood forecasting/480m"}


# ------------------------------------------------------------------------------------------------- windows
def scan_wet_windows(rd, positions, starts, patch=PATCH, wet_m=0.05):
    """(row, col, start) for windows whose patch is wet (>= wet_m) at the last observed OR the last forecast frame.
    Dry windows carry no signal (persistence is exact) and would only dilute the metrics."""
    out = []
    for st in starts:
        hist, fut = rd.window_frame_indices(int(st), T_IN, HORIZON, STRIDE)
        with rasterio.open(rd.depth_files[hist[-1]]) as s:
            last = s.read(1)
        with rasterio.open(rd.depth_files[fut[-1]]) as s:
            end = s.read(1)
        for (r, c) in positions:
            sl = (slice(r, r + patch), slice(c, c + patch))
            if max(float(last[sl].max()), float(end[sl].max())) >= wet_m:
                out.append((r, c, int(st)))
    return out


def load_examples(rd, windows, patch=PATCH):
    ex = []
    for (r, c, st) in windows:
        x, y, _, _ = rd.load_patch_sequence(r, c, patch, st, T_IN, HORIZON, STRIDE)
        ex.append((x, y, np.ones((patch, patch), bool)))
    return ex


def load_padded_examples(rd, positions, starts, size, wet_m=0.05):
    """For a domain smaller than one patch in some direction (UK at 60 m is 85x137): read the largest square that fits, pad it
    to 128x128 (depth 0; terrain and rain edge-replicated) and return a mask of the real pixels, so scoring ignores the padding."""
    ex = []
    for st in starts:
        for (r, c) in positions:
            x, y, _, mx = rd.load_patch_sequence(r, c, size, int(st), T_IN, HORIZON, STRIDE)
            if max(float(x[-1, ..., 0].max()), float(y[-1].max())) < wet_m:
                continue
            p = PATCH - size
            xp = np.pad(x, ((0, 0), (0, p), (0, p), (0, 0)), mode="edge")
            xp[:, size:, :, 0] = 0.0
            xp[:, :, size:, 0] = 0.0
            yp = np.pad(y, ((0, 0), (0, p), (0, p)))
            mask = np.zeros((PATCH, PATCH), bool)
            mask[:size, :size] = True
            ex.append((xp, yp, mask))
    return ex


# ------------------------------------------------------------------------------------------------- model
class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Linear warm-up to `peak` over `warmup_steps`, then cosine decay to alpha*peak. The 60 m Australia run collapsed to an
    input-independent network within ~100 steps at a constant-start LR of 1e-3 (Adam + a few extreme windows); warm-up removes that."""

    def __init__(self, peak, total_steps, warmup_steps, alpha=0.02):
        self.peak, self.total, self.warm, self.alpha = float(peak), float(total_steps), float(max(1, warmup_steps)), float(alpha)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warm = self.peak * (step + 1.0) / self.warm
        prog = tf.clip_by_value((step - self.warm) / tf.maximum(1.0, self.total - self.warm), 0.0, 1.0)
        cos = self.peak * (self.alpha + (1.0 - self.alpha) * 0.5 * (1.0 + tf.cos(np.pi * prog)))
        return tf.where(step < self.warm, warm, cos)

    def get_config(self):
        return dict(peak=self.peak, total_steps=self.total, warmup_steps=self.warm, alpha=self.alpha)


def prediction_spread(predict, examples, n=12):
    """Std across windows of the predicted mean change (metres). ~0 means the network answers the same for every input (dead)."""
    d = [(predict(x) - x[-1, ..., 0][None]).mean() for x, _, _ in examples[:n]]
    return float(np.std(d))


def build_and_train(train, val, epochs, seed, lr=1e-3, clipnorm=1.0, warmup_steps=200, log=print):
    tf.keras.utils.set_random_seed(seed)
    encoder = flood_model.build_encoder(hidden_dim=128, dropout=0.3)
    decoder = flood_model.build_decoder_step(hidden_dim=128, dropout=0.3)
    pers_mse = max(float(np.mean([np.mean((y - x[-1, ..., 0][None]) ** 2) for x, y, _ in train])), 1e-12)
    sched = WarmupCosine(lr, epochs * len(train), warmup_steps)
    opt = flood_model.OPTIMIZERS["adam"](learning_rate=sched, clipnorm=clipnorm)
    vars_ = encoder.trainable_variables + decoder.trainable_variables
    xs = tf.TensorSpec([1, T_IN, PATCH, PATCH, 3], tf.float32)
    ys = tf.TensorSpec([HORIZON, PATCH, PATCH], tf.float32)

    @tf.function(input_signature=[xs, ys])
    def step(x, y):
        with tf.GradientTape() as tape:
            frames = decode_horizon(decoder, encoder(x, training=True), x, HORIZON, training=True)
            loss = tf.reduce_mean(tf.square(frames[0] - y)) / pers_mse
        g = tape.gradient(loss, vars_)
        opt.apply_gradients((gi, v) for gi, v in zip(g, vars_) if gi is not None)
        return loss

    @tf.function(input_signature=[xs])
    def pred(x):
        return decode_horizon(decoder, encoder(x, training=False), x, HORIZON, training=False)[0]

    def predict(x):
        return pred(tf.constant(x[None], tf.float32)).numpy()

    def val_skill():
        return skill(all_preds(predict, val))

    log(f"persistence MSE on training windows {pers_mse:.3e} m^2 (RMSE {np.sqrt(pers_mse) * 1000:.1f} mm); "
        f"epoch 0 val skill {val_skill():+.4f} (must be ~0)")
    hist = []
    for ep in range(1, epochs + 1):
        t0 = time.perf_counter()
        losses = []
        for i in np.random.permutation(len(train)):
            x, y, _ = train[i]
            losses.append(float(step(tf.constant(x[None], tf.float32), tf.constant(y, tf.float32))))
        sk = val_skill()
        hist.append((ep, float(np.mean(losses)), sk))
        spread = prediction_spread(predict, val)
        if ep >= 2 and spread < 1e-5:
            raise RuntimeError(f"DEAD NETWORK after epoch {ep}: the predicted change is identical for every validation window "
                               f"(std {spread:.2e} m). Its skill would be an input-independent constant offset, not a result.")
        log(f"epoch {ep:2d}/{epochs}  train/persist {np.mean(losses):.4f}  val skill {sk:+.4f}  ({time.perf_counter() - t0:.1f}s)")
    return predict, hist, pers_mse


# ------------------------------------------------------------------------------------------------- evaluation
def all_preds(predict, examples, transform=None):
    P, O, Q, M = [], [], [], []
    for x, y, m in examples:
        P.append(predict(x if transform is None else transform(x)))
        O.append(y)
        Q.append(np.repeat(x[-1, ..., 0][None], HORIZON, axis=0))
        M.append(np.broadcast_to(m, (HORIZON,) + m.shape))
    return np.stack(P), np.stack(O), np.stack(Q), np.stack(M)


def skill(t):
    p, o, q, m = t
    return 1.0 - np.mean(((p - o) ** 2)[m]) / np.mean(((q - o) ** 2)[m])


def metrics(p, o, m, wet):
    pp, oo = p[m], o[m]
    rmse = float(np.sqrt(np.mean((pp - oo) ** 2)))
    nse = float(1 - np.sum((oo - pp) ** 2) / np.sum((oo - oo.mean()) ** 2))
    r = float(np.corrcoef(pp, oo)[0, 1])
    pw, ow = pp > wet, oo > wet
    csi = float((pw & ow).sum() / max(1, (pw | ow).sum()))
    return dict(RMSE=rmse, NSE=nse, r=r, CSI=csi)


def report(name, predict, examples, log=print):
    p, o, q, m = all_preds(predict, examples)
    row = {"n_windows": len(examples), "skill": float(skill((p, o, q, m)))}
    row["model"] = {f"CSI@{w}": metrics(p, o, m, w) for w in (0.001, 0.1)}
    row["persistence"] = {f"CSI@{w}": metrics(q, o, m, w) for w in (0.001, 0.1)}
    lead = [float(1 - np.mean(((p[:, k] - o[:, k]) ** 2)[m[:, k]]) / np.mean(((q[:, k] - o[:, k]) ** 2)[m[:, k]])) for k in range(HORIZON)]
    pr = [float(np.sqrt(np.mean(((q[:, k] - o[:, k]) ** 2)[m[:, k]])) * 1000) for k in range(HORIZON)]
    row["lead_skill"], row["persistence_rmse_mm"] = lead, pr
    a, b = row["model"]["CSI@0.001"], row["persistence"]["CSI@0.001"]
    log(f"\n{name}: {len(examples)} windows")
    log(f"  skill vs persistence {row['skill']:+.4f}")
    log(f"  RMSE  model {a['RMSE']:.4f} m   persistence {b['RMSE']:.4f} m")
    log(f"  NSE   model {a['NSE']:.4f}     persistence {b['NSE']:.4f}      r  model {a['r']:.4f}  persistence {b['r']:.4f}")
    log(f"  CSI(0.001 m) model {a['CSI']:.4f} persistence {b['CSI']:.4f};  CSI(0.1 m) model {row['model']['CSI@0.1']['CSI']:.4f} "
        f"persistence {row['persistence']['CSI@0.1']['CSI']:.4f}")
    log("  skill by lead time (30 min ... 2.5 h): " + "  ".join(f"{v:+.3f}" for v in lead) +
        "   | persistence RMSE mm: " + " ".join(f"{v:.1f}" for v in pr))
    return row


def ablations(name, predict, examples, log=print):
    def no_rain(x):
        y = x.copy(); y[..., 2] = 0.0; return y

    def flat_dem(x):
        y = x.copy(); y[..., 1] = 0.0; return y

    def scrambled(x):
        return np.concatenate([x[:-1][::-1], x[-1:]], axis=0)

    def frozen(x):
        return np.repeat(x[-1:], x.shape[0], axis=0)

    out = {}
    log(f"\n{name}: what does the model use? (skill when one input is removed at test time)")
    for label, fn in (("none (normal)", None), ("rainfall removed", no_rain), ("terrain removed", flat_dem),
                      ("history order scrambled", scrambled), ("history frozen", frozen)):
        out[label] = float(skill(all_preds(predict, examples, fn)))
        log(f"    {label:26s} {out[label]:+.3f}")
    return out


# ------------------------------------------------------------------------------------------------- the two experiments
def experiment(kind, epochs=40, n_train=400, n_val=150, n_starts=80, seed=42, log=print):
    rng = np.random.default_rng(seed)
    if kind == "pakistan_mozambique":
        res_train, res_test, tr_name, te_name = 480, 480, "Pakistan", "Mozambique"
    elif kind == "australia60_uk60":
        res_train, res_test, tr_name, te_name = 60, 60, "Australia", "UK"
    else:
        raise SystemExit(f"unknown experiment {kind}")
    rd_tr = RegionData(root_dir=ROOT, region=tr_name, fidelity_dir=FID[res_train])
    rd_te = RegionData(root_dir=ROOT, region=te_name, fidelity_dir=FID[res_test])
    log(f"=== {tr_name} ({res_train} m, depth {rd_tr.grid_height}x{rd_tr.grid_width}) -> {te_name} ({res_test} m, depth "
        f"{rd_te.grid_height}x{rd_te.grid_width}); epochs {epochs}, seed {seed}")
    pos = valid_positions(rd_tr, PATCH, 1.0)
    perm = rng.permutation(len(pos))
    nv = max(1, int(round(len(pos) * 0.25)))
    val_pos = [pos[i] for i in perm[:nv]]
    tr_pos = [pos[i] for i in perm[nv:]]
    assert not set(val_pos) & set(tr_pos)
    log(f"{tr_name}: {len(pos)} valid {PATCH}px tiles -> {len(tr_pos)} train / {len(val_pos)} validation (split by tile)")
    starts = start_indices_for(rd_tr, n_starts, T_IN, HORIZON, STRIDE)
    tr_pool = scan_wet_windows(rd_tr, tr_pos, starts)
    va_pool = scan_wet_windows(rd_tr, val_pos, starts)
    log(f"wet windows available: train {len(tr_pool)}, validation {len(va_pool)}")
    tr_w = [tr_pool[i] for i in rng.choice(len(tr_pool), size=min(n_train, len(tr_pool)), replace=False)]
    va_w = [va_pool[i] for i in rng.choice(len(va_pool), size=min(n_val, len(va_pool)), replace=False)]
    train, val = load_examples(rd_tr, tr_w), load_examples(rd_tr, va_w)
    log(f"loaded {len(train)} training and {len(val)} validation windows")

    # test region
    te_starts = start_indices_for(rd_te, 60, T_IN, HORIZON, STRIDE)
    small = min(rd_te.grid_height, rd_te.grid_width) < PATCH
    if small:                                        # the domain is smaller than a patch in one direction: padded + masked
        size = min(rd_te.grid_height, rd_te.grid_width)
        cols = sorted({0, rd_te.grid_width - size})
        rows = sorted({0, rd_te.grid_height - size})
        test = load_padded_examples(rd_te, [(r, c) for r in rows for c in cols], te_starts, size)
        log(f"{te_name}: domain {rd_te.grid_height}x{rd_te.grid_width} < one patch -> {size}px windows padded to {PATCH}, scored on real pixels only")
    else:
        rows = sorted({0, rd_te.grid_height - PATCH})
        cols = sorted({0, rd_te.grid_width - PATCH})
        tw = scan_wet_windows(rd_te, [(r, c) for r in rows for c in cols], te_starts)
        test = load_examples(rd_te, tw)
        log(f"{te_name}: {len(rows) * len(cols)} (overlapping) {PATCH}px tiles x {len(te_starts)} times -> {len(test)} wet windows")

    predict, hist, pers = build_and_train(train, val, epochs, seed, log=log)
    out = {"experiment": kind, "seed": seed, "epochs": epochs, "n_train": len(train), "n_val": len(val), "n_test": len(test),
           "history": hist, "persistence_mse_train": pers}
    out["validation"] = report(f"{tr_name} held-out tiles", predict, val, log)
    out["test"] = report(f"{te_name} (never seen)", predict, test, log)
    out["ablation_validation"] = ablations(f"{tr_name} held-out tiles", predict, val, log)
    out["ablation_test"] = ablations(f"{te_name} (never seen)", predict, test, log)
    return out


if __name__ == "__main__":
    kind = sys.argv[1]
    epochs = int(sys.argv[2]) if len(sys.argv) > 2 else 40
    seed = int(sys.argv[3]) if len(sys.argv) > 3 else 42
    n_train = int(sys.argv[4]) if len(sys.argv) > 4 else 400
    res = experiment(kind, epochs=epochs, seed=seed, n_train=n_train, n_val=min(150, n_train // 2), n_starts=80 if n_train >= 100 else 20)
    os.makedirs("verification_outputs/multires", exist_ok=True)
    path = f"verification_outputs/multires/{kind}_seed{seed}_ep{epochs}.json"
    json.dump(res, open(path, "w"), indent=1, default=float)
    print("\nsaved", path)


Writing depth_experiment.py


## Check the geometry loads

Prints, for each region and resolution used, the depth grid, the registered scale (30 m base pixels per depth cell) and offset, and
how many 128-px tiles are usable. A region without a registered geometry is refused, never guessed.

In [ ]:
import sys
sys.path.insert(0, ".")
import depth_experiment as de
from real_data_loader import RegionData
from windows import valid_positions, all_positions

de.ROOT = DATA_ROOT
NEEDED = []
if "pakistan_mozambique" in EXPERIMENTS:
    NEEDED += [("Pakistan", 480), ("Mozambique", 480)]
if "australia60_uk60" in EXPERIMENTS:
    NEEDED += [("Australia", 60), ("UK", 60)]
for region, res in NEEDED:
    rd = RegionData(root_dir=DATA_ROOT, region=region, fidelity_dir=de.FID[res])
    print(f"{region:10s} {res:3d} m  depth {rd.grid_height}x{rd.grid_width}  scale {rd.layer_scale}  offset {rd.layer_offset}  "
          f"{len(rd.depth_files)} frames  tiles: {len(all_positions(rd, 128))} -> valid {len(valid_positions(rd, 128, 1.0))}")

Pakistan   480 m  depth 810x441  scale 16  offset (42, 178)  4033 frames  tiles: 18 -> valid 18
Mozambique 480 m  depth 151x138  scale 16  offset (150, 50)  1729 frames  tiles: 1 -> valid 1
Australia   60 m  depth 536x536  scale 2  offset (0, 0)  2881 frames  tiles: 16 -> valid 16
UK          60 m  depth 85x137  scale 2  offset (5, 5)  865 frames  tiles: 0 -> valid 0


## Run the experiments

Each one: split the training region's tiles into train / validation (no shared tile), train the persistence-anchored CNN-LSTM on
depth only (loss = MSE / persistence MSE, so 1.0 means exactly copying the last frame), then score the held-out tiles and the test
region. A test domain smaller than one patch (the UK at 60 m is 85x137) is padded and scored on its real pixels only.

In [ ]:
results = {}
for name in EXPERIMENTS:
    results[name] = de.experiment(name, epochs=EPOCHS, n_train=N_TRAIN, n_val=N_VAL, n_starts=N_STARTS, seed=SEED)

=== Pakistan (480 m, depth 810x441) -> Mozambique (480 m, depth 151x138); epochs 40, seed 42
Pakistan: 18 valid 128px tiles -> 14 train / 4 validation (split by tile)
wet windows available: train 1120, validation 320
loaded 400 training and 150 validation windows
Mozambique: 4 (overlapping) 128px tiles x 60 times -> 238 wet windows
persistence MSE on training windows 1.042e-04 m^2 (RMSE 10.2 mm); epoch 0 val skill +0.0000 (must be ~0)
epoch  1/40  train/persist 0.9209  val skill +0.0993  (21.3s)
epoch  2/40  train/persist 0.9043  val skill +0.1121  (10.6s)
epoch  3/40  train/persist 0.8798  val skill +0.1187  (10.4s)
epoch  4/40  train/persist 0.8814  val skill +0.0925  (10.4s)
epoch  5/40  train/persist 0.8661  val skill +0.1197  (9.9s)
epoch  6/40  train/persist 0.8707  val skill +0.1216  (10.4s)
epoch  7/40  train/persist 0.8889  val skill +0.0968  (10.4s)
epoch  8/40  train/persist 0.8600  val skill +0.1141  (10.8s)
epoch  9/40  train/persist 0.8663  val skill +0.1223  (10.4s)
epoc

## Summary

Skill above 0 beats persistence. The paper's numbers are shown for orientation only (a different task: 19 steps from the initial state,
whole domain); the persistence RMSE on OUR windows is printed so the scale of the problem is visible.

In [ ]:
import json
print(f"{'experiment':22s} {'held-out skill':>15s} {'test skill':>11s} {'test RMSE model / persistence (m)':>36s}")
for name, r in results.items():
    a = r["test"]["model"]["CSI@0.001"]["RMSE"]; b = r["test"]["persistence"]["CSI@0.001"]["RMSE"]
    print(f"{name:22s} {r['validation']['skill']:+15.4f} {r['test']['skill']:+11.4f} {a:15.4f} / {b:.4f}")
print()
print("Paper (Xu et al. 2025, Table 5, different protocol): Pakistan->Mozambique 480 m FNO+ RMSE 0.0786 m, NSE 0.955 (U-Net RMSE 1.60, NSE -72);"
      " Australia->UK 60 m FNO+ RMSE 0.0248 m, NSE 0.9989 (U-Net RMSE 0.525).")
json.dump(results, open("multires_results.json", "w"), indent=1, default=float)
print("saved multires_results.json")

experiment              held-out skill  test skill    test RMSE model / persistence (m)
pakistan_mozambique            +0.1187     +0.0676          0.0165 / 0.0171
australia60_uk60               +0.4112     +0.1640          0.0144 / 0.0157

Paper (Xu et al. 2025, Table 5, different protocol): Pakistan->Mozambique 480 m FNO+ RMSE 0.0786 m, NSE 0.955 (U-Net RMSE 1.60, NSE -72); Australia->UK 60 m FNO+ RMSE 0.0248 m, NSE 0.9989 (U-Net RMSE 0.525).
saved multires_results.json
